# ノートブックを Drive に置く（DP9 本番）

リポジトリの決まったコミット（`4602a486adbf`）から、次のノートブックを Drive の `DECISION_POINTS` にそのままコピーします。1 つずつ sha256 で中身が同じかを確かめます。**API も GPU も使いません（費用 $0）**。

置くもの：`DP9_QWEN_A.ipynb`、`DP9_QWEN_B.ipynb`、`DP9_QWEN_C.ipynb`、`DP9_QWEN_D.ipynb`、`DP9_ANALYZE.ipynb`

> 🔑 **先に許可しておくシークレット**：`GITHUB_TOKEN`（必須）

**使い方**：CPU のランタイム。「すべてのセルを実行」で 1〜2 分。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。
- **エラーのとき**：通知の本文にエラーの文面（トレースバック）が入ります。全文は Drive の `DECISION_POINTS/_errors/` にも保存されるので、Claude が直接読めます。

In [ ]:
# Cell 1 — copy the listed notebooks from the repo (exact commit) into Drive, each checked by sha256.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, time, hashlib, datetime as dt

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。エラーのときはエラーの文面も送ります（鍵・トークンらしい文字は伏せ字。全文は Drive の DECISION_POINTS/_errors）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'ノートブックを置く（DP9 本番）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'GodPork - place notebooks', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
import os, re as _re, traceback as _tb, datetime as _edt
ERR_SLUG = 'PLACE_NOTEBOOKS'   # Drive のエラーファイルの名前の頭
_SECRET_RE = _re.compile(r"sk-ant-[A-Za-z0-9_\-]+|github_pat_[A-Za-z0-9_]+|gh[pousr]_[A-Za-z0-9]{8,}|hf_[A-Za-z0-9]{8,}|x-access-token:[^@\s]+@")
def _scrub(s):
    s = _SECRET_RE.sub('[伏せ字]', s)
    return '\n'.join(l if len(l) <= 300 else l[:300] + ' …（長いので省略）' for l in s.splitlines())
def _error_text(result):
    e = result.error_in_exec if result.error_in_exec is not None else result.error_before_exec
    tb = ''.join(_tb.format_exception(type(e), e, e.__traceback__))
    first = (getattr(getattr(result, 'info', None), 'raw_cell', '') or '').strip().splitlines()[:1]
    return _scrub('セル: ' + (first[0][:100] if first else '?') + '\n' + tb)
def _save_error(text):
    base = '/content/drive/MyDrive'
    if not os.path.isdir(base):
        return None
    d = os.path.join(base, 'DECISION_POINTS', '_errors')
    os.makedirs(d, exist_ok=True)
    stem = ERR_SLUG + '_' + _edt.datetime.now(_edt.timezone(_edt.timedelta(hours=9))).strftime('%Y%m%d_%H%M%S')
    name, n = stem + '.txt', 1
    while os.path.exists(os.path.join(d, name)):
        n += 1
        name = stem + '_' + str(n) + '.txt'
    with open(os.path.join(d, name), 'w', encoding='utf-8') as fh:
        fh.write('[' + NB_LABEL + ']\n' + text)
    return 'DECISION_POINTS/_errors/' + name
def _short(text, limit=3300):
    b = text.encode('utf-8')
    if len(b) <= limit:
        return text
    return '…（前のほうは省略。全文は Drive）\n' + b[-limit:].decode('utf-8', 'ignore')
def _on_error(result):
    if result.error_in_exec is None and result.error_before_exec is None:
        return
    try:
        text = _error_text(result)
    except Exception as e2:
        text = 'エラーの文面を作れませんでした：' + type(e2).__name__
    try:
        where = _save_error(text)
    except Exception as e3:
        where = None
        text += '\n（Drive に保存できませんでした：' + type(e3).__name__ + '）'
    print('エラーの全文を通知に送ります' + ('（Drive：' + where + '）' if where else ''))
    notify('⚠ 止まりました：エラーが出ています。' + ('全文は Drive の ' + where if where else 'Drive には保存できませんでした')
           + '\n\n' + _short(text), tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

drive.mount("/content/drive")

COMMIT = "4602a486adbf3248b8542059f16dde6c00c6494b"
FILES = json.loads(r"""{"notebooks/DP9_QWEN_A.ipynb": "867f3e8b87819c5cf8e07751f237be242bfbbef19f2ceef94cf97bd7ceff5a57", "notebooks/DP9_QWEN_B.ipynb": "21c455a904826b0b4465af6d1d3314137e5e9cf55450948bf163204c5e3e2862", "notebooks/DP9_QWEN_C.ipynb": "f04437ed2a29b158b7941f17ee293a50ec12131f66aeb9ebc301629f59385f37", "notebooks/DP9_QWEN_D.ipynb": "e5e101b1df6fb3a02ae3edea0610427dcd20aa6501b7a631c99c006059323091", "notebooks/DP9_ANALYZE.ipynb": "172ec9ff5017613700af72a41ac5de3006d45d497936bb0299cb9ea1d3ef813c"}""")          # repo path -> sha256
DEST = Path("/content/drive/MyDrive") / "DECISION_POINTS"
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork_place")
try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
assert token and str(token).strip(), "Colab のシークレットに GITHUB_TOKEN を登録してね（ノートブックへのアクセスもオン）。"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=str(token).strip(), GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
token = None
os.chdir('/content')   # セル1 をやり直しても、消すフォルダの中にいないように（2026-10-14）
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "-q", "--filter=blob:none", "--branch", "decision-points", "--single-branch", REPO_URL,
                str(REPO)], check=True, env=env)
# The partial clone fetches older blobs on checkout, so the token must still be set here (fix 2026-10-17: a commit
# that is not the branch tip failed with exit 128 when the token had already been removed).
subprocess.run(["git", "-C", str(REPO), "checkout", "-q", "--detach", COMMIT], check=True, env=env)
env.pop("GITHUB_TOKEN", None)
DEST.mkdir(parents=True, exist_ok=True)
stamp = dt.datetime.now(dt.timezone(dt.timedelta(hours=9))).strftime("%Y%m%d_%H%M%S")
placed = []
for rel, want in FILES.items():
    data = (REPO / rel).read_bytes()
    assert hashlib.sha256(data).hexdigest() == want, ("repo file differs from the expected hash", rel)
    target = DEST / Path(rel).name
    if target.exists():
        if hashlib.sha256(target.read_bytes()).hexdigest() == want:
            placed.append((target.name, "すでに同じものがあります"))
            continue
        target.rename(target.with_name(target.stem + "_superseded_" + stamp + target.suffix))   # kept, not deleted
    tmp = target.with_name(target.name + ".tmp")
    tmp.write_bytes(data)
    os.replace(tmp, target)
    assert hashlib.sha256(target.read_bytes()).hexdigest() == want, ("Drive copy differs", rel)
    placed.append((target.name, "置きました"))
shutil.rmtree(REPO)
for name, what in placed:
    print(name, "→", what)
print("✅ 置き終わりました（場所：MyDrive/DECISION_POINTS）。Claude に知らせてね")
notify('🎉 終わりました：ノートブックを Drive に置きました（DP9 本番）', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()